In [1]:
#I am running this mostly because it was giving me errors
#%pip uninstall streamlit -y
#run this if you need to instale gradio still otherwise keep it like this
#%pip install --upgrade gradio
#%pip install --upgrade Pillow
#This was for error in the next one
#%pip install ipywidgets
#%pip install gradio-modal

In [2]:
#The imports that were used
import gradio as gr
from PIL import Image, ImageDraw
import os
from os import listdir
from pathlib import Path
import numpy as np
import random
import pandas as pd
from pathlib import Path
from gradio_modal import Modal
#This will be where we then put in the images. Create a new block under this with the two sets and stuff to keep track of 
#which ones we have had and which ones we did not
home = Path.home()
#folder_img = home / "OneDrive" / "Documenten" / "PKG - CDD-CESM" / "CDD-CESM" / "Subtracted images of CDD-CESM"  # DIT WAS DE OG
folder_img = Path(r"C:\Program Files\PKG - CDD-CESM\CDD-CESM\Subtracted images of CDD-CESM")  # EN DEZE IS DE MIJNE
#folder_img = Path(r"C:\Users\mulle\models_machine_learning\CLEANED\CDD-CESM\Subtracted images of CDD-CESM")
# Path for handmade ellipse mask
mask_folder = Path.cwd() / "Manual_masks"
mask_folder.mkdir(exist_ok=True)

# Empty for every new session
for old_mask in mask_folder.glob("*.jpg"):
    old_mask.unlink()

#Threshold for the size of the image
size_threshold = 1500

In [3]:
#This block is for defining excel functions and the radio check functions


def load_excel_annotations(xlsx_path="Radiology-manual-annotations.xlsx"):
    """
    This is the dataprep for the data from the excel sheet
    """
    df = pd.read_excel(xlsx_path, usecols=["Image_name", "Pathology Classification/ Follow up"])
    df.columns = ["image_name", "pathology"]
    return dict(zip(df["image_name"], df["pathology"]))



def check_annotation(radio_choice):
    """
    Return elipse overlap and if the chosen classification is correct
    """
    if not current_img:
        return "No image loaded!"
    
    filename = os.path.basename(current_img[-1]).replace(".jpg", "")
    expected = excel_annotations.get(filename, None)

    #Check if classification is correct compared to the doctors classfication
    if expected is None:
        return f"'{filename}' not found in Excel."
    
    if radio_choice.strip().lower() == expected.strip().lower():
        result = f"Tumor classification: Correct"  
        # check_results[current_img[-1]] = "Tumor classification: Correct." 
    else:
        result = f"Tumor classification: Incorrect."
        # check_results[current_img[-1]] = "Tumor classification: Incorrect."
    
    # Add overlap result if any ellipses were drawn for this image
    if radio_choice != "Normal":
        for i, ann in enumerate(end_coords.get(current_img[-1], []), 1):
            overlap_str, _ = check_ellipse_overlap(current_img[-1], ann)
            result += f"\nEllipse {i} overlap: {overlap_str}"
            
    check_results[current_img[-1]] = result 
    saved_radio_choices[current_img[-1]] = radio_choice
    return result  


def show_annotation():
    """
    Tell the user the right classification
    """
    if not current_img:
        return "No image loaded!"
        
    filename = os.path.basename(current_img[-1]).replace(".jpg", "")
    expected = excel_annotations.get(filename, None)
    
    if expected is None:
        return f"'{filename}' not found in Excel."
        
    if expected.strip().lower() == "normal":
        return f"The correct answer is: '{expected}'. This scan is normal, so there are no tumors to see."
    else:
        return f"The correct answer is: '{expected}', click the button 'Show tumor area' to see where tumors are."

In [4]:
#So this is for looping throug images in folder_img

# Filtered out cases that don't have masks
#mask_folder_groundtruth = Path(r"C:\Users\aryan\SysteemProject\SysteemProject\masks_subtracted")
mask_folder_groundtruth = Path.cwd() / "masks_subtracted"
excel_annotations = load_excel_annotations("Radiology-manual-annotations.xlsx")
images_path = []

for images in os.listdir(folder_img):
    if images.lower().endswith(".jpg"):
        image_name = Path(images).stem
        mask_path = mask_folder_groundtruth / f"{image_name}_subtracted_mask.jpg"
        # Gets the classification from the excel_annotations
        expected_class = excel_annotations.get(image_name, "").strip().lower()
         # Adds if there is a mask OR if its a Normal case
        if mask_path.exists() or expected_class == "normal":
            images_path.append(folder_img / images)

#Randomize the images
random.shuffle(images_path)
#Set a max to the list of images
images_path = images_path[:5]
#Also keep track of which images we have had
clicked = []
#Keep tack on what image is showing now
current_img = []
#A dictionary with the results of the type and image
check_results = {}
#Keeps track of the skipped images
skipped_imgs = set()
#Keep track of radio button choice per image
saved_radio_choices = {}

coords_history = {}

In [5]:
#This block is for the loading of images


def load_image ():
    """
    This will simily load the image so the person can click on it
    """
    #reset values (not the saved ones)
    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None
    
    #First check if list is not empty
    if not images_path:
        return None

    #Pop from the list with images and add it to the set clicked
    path = images_path.pop()
    current_img.append(path)
    

    #reduce size of image to reduce processing time
    img = Image.open(path)
    new_size = get_scaled_dimensions(img)
    img = img.resize(new_size, Image.Resampling.LANCZOS)
    
    return redraw_existing_annotations(img, path)

def next_image ():
    """
    Load the next image
    """
    current_path = current_img[-1]
    # If there has been no check for the img → skipped
    if current_path not in check_results:
        skipped_imgs.add(current_path)
    #add the current image to clicked
    clicked.append(current_img.pop())
    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None
    
    if not images_path:
        return None, gr.update(value="Benign")

    path = images_path.pop()
    current_img.append(path)
    
    img_radio_choice = saved_radio_choices.get(path, "Benign")

    img = Image.open(path)
    new_size = get_scaled_dimensions(img)
    img = img.resize(new_size, Image.Resampling.LANCZOS)
    
    color = "gray" if img_radio_choice == "Normal" else "green" if img_radio_choice == "Benign" else "red"
    img_drawn = redraw_existing_annotations(img, path, color=color)
    
    # Restore saved result for this image, or clear it
    saved = check_results.get(current_img[-1], "") if current_img else ""
    
    return img_drawn, gr.update(visible=bool(clicked)), gr.update(visible=bool(images_path)), saved, gr.update(value=img_radio_choice)

def back_image ():
    """
    Load the previous image
    """
    #Check if there is an old image
    if not clicked:
        return gr.Warning("No previous images!")

    current_path = current_img[-1]
    if current_path not in check_results:
        skipped_imgs.add(current_path)
        
    #reset current clicks
    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None
    #add the photo back to the other list with unseen images
    images_path.append(current_img.pop())
    #get previous image of clicked images
    path = clicked.pop()
    current_img.append(path)
    img_radio_choice = saved_radio_choices.get(path, "Benign")
    img_raw = Image.open(path)
    new_size = get_scaled_dimensions(img_raw)
    img_raw = img_raw.resize(new_size, Image.Resampling.LANCZOS)

    #redraw annotations on the right scale
    color = "gray" if img_radio_choice == "Normal" else "green" if img_radio_choice == "Benign" else "red"
    img = redraw_existing_annotations(img_raw, path, color)
    saved = check_results.get(current_img[-1], "")
    if current_img and current_img[-1] in coords_history:
        saved_coords = coords_history.get(current_img[-1], "No annotations for this image.")
    else:
        saved_coords = ""

    #Load drawn masks if they exists, otherwise load a new empty mask
    name = Path(path).stem
    manual_mask_path = mask_folder / f"{name}_mask.jpg"
    if manual_mask_path.exists():
        updated_mask = np.array(Image.open(manual_mask_path).convert("RGB"))
    else:
        updated_mask = np.zeros((new_size[1], new_size[0], 3), dtype=np.uint8)

    return img, gr.update(visible=bool(clicked)), gr.update(visible=bool(images_path)), saved, saved_coords, updated_mask, gr.update(value=img_radio_choice)

def get_progress():
    """
    Function for the progressbar
    """
    #Check how many images there are
    total = len(clicked) + len(images_path) + len(current_img)
    if total == 0:
        return "No images loaded."

    #Check how many images have already been checked
    checked = set(check_results.keys())
    done = min(len(checked), total)

    #Make the progressbar visible
    percent = int((done / total) * 100)
    bar_length = 65
    filled_length = int(bar_length * percent / 100)
    bar = "█" * filled_length + "░" * (bar_length - filled_length)
    
    correct = 0
    wrong = 0

    #If the classification is correct and the dice score is higher or the same as 0.5 (our chosen threshold), the answer will be classified as correct. 
    #If one of them does not validate, the answer will be classified as incorrect and if the user did not check their mask or classification, the image will be classified as skipped
    for path, result_text in check_results.items():
        if "Tumor classification: Correct" in result_text:
            filename = os.path.basename(path).replace(".jpg", "")
            expected_class = excel_annotations.get(filename, "").strip().lower()
            
            if expected_class == "normal":
                correct += 1
            else:
                annotations = end_coords.get(path, [])
                if not annotations:
                    wrong += 1
                    continue
                    
                highest_dice = 0.0
                for ann in annotations:
                    _, dice_val = check_ellipse_overlap(path, ann) 
                    if dice_val > highest_dice:
                        highest_dice = dice_val
                        
                if highest_dice >= 0.5:
                    correct += 1
                else:
                    wrong += 1
        else:
            wrong += 1
            
    skipped = sum(1 for p in skipped_imgs if p not in check_results)
    
    return (
        f"Progress: [{bar}] {done}/{total} images ({percent}%)\n"
        f"✅ Correct: {correct}  |  ❌ Wrong: {wrong}  |  ⏭️ Skipped: {skipped}"
    )


def restart_session():
    """
    Restart the whole session of the application for new images and reset all saved information of this session
    """
    # Reset all global states
    global images_path, clicked, check_results, skipped_imgs
    hint_index.clear()
    clicked.clear()
    current_img.clear()
    end_coords.clear()
    check_results.clear()
    skipped_imgs.clear()
    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None
    saved_radio_choices.clear()

    # Empty manual masks map
    for old_mask in mask_folder.glob("*.jpg"):
        old_mask.unlink()

    # Load all images again
    images_path = []
    for images in os.listdir(folder_img):
        if images.lower().endswith(".jpg"):
            image_name = Path(images).stem
            mask_path = mask_folder_groundtruth / f"{image_name}_subtracted_mask.jpg"
            
            expected_class = excel_annotations.get(image_name, "").strip().lower()
            if mask_path.exists() or expected_class == "normal":
                images_path.append(folder_img / images)
    random.shuffle(images_path)
    images_path = images_path[:5]

    first = load_image()
    return (
        first,
        gr.update(visible=False),   # back_btn
        gr.update(visible=True),    # next_btn
        gr.update(visible=True),   # restart_btn
        "", "", "", get_progress(), "",
        "Benign"
    )

def show_masks():
    """
    Show the mask of the user (manual mask) next to the correct answer (ground truth mask) that the model predicted
    """
    if not current_img:
        return None, None, gr.update(visible=True), gr.update(visible=False)
    path = current_img[-1]
    name = Path(path).stem
    # gets image dimensions
    img_raw = Image.open(path)
    new_size = get_scaled_dimensions(img_raw)
    # Handmade mask or empty black mask for normal imagtes
    manual_mask_path = mask_folder / f"{name}_mask.jpg"
    if manual_mask_path.exists():
        manual = np.array(Image.open(manual_mask_path).convert("RGB"))
    else:
        manual = np.zeros((new_size[1], new_size[0], 3), dtype=np.uint8)
    # Ground truth masks
    gt_mask_path = mask_folder_groundtruth / f"{name}_subtracted_mask.jpg"
    if gt_mask_path.exists():
        gt = np.array(Image.open(gt_mask_path).convert("RGB"))
    else:
        # For normal images, make a black mask with nothing on it. 
        gt = np.zeros((new_size[1], new_size[0], 3), dtype=np.uint8)

    return manual, gt, gr.update(visible=True), gr.update(visible=True)

def get_current_gt_mask():
    """
    Get the ground truth mask of the current image
    """
    if not current_img:
        return np.zeros((100, 100, 3), dtype=np.uint8)
    path = current_img[-1]
    name = Path(path).stem
    gt_mask_path = mask_folder_groundtruth / f"{name}_subtracted_mask.jpg"
    if gt_mask_path.exists():
        return np.array(Image.open(gt_mask_path).convert("RGB"))
    return np.zeros((100, 100, 3), dtype=np.uint8)


def get_current_manual_mask():
    """
    Get the drawn mask of the user of the current image (This one is always black when restarting or after resetting the annotations)
    """
    if not current_img:
        return np.zeros((100, 100, 3), dtype=np.uint8)
    path = current_img[-1]
    name = Path(path).stem
    manual_mask_path = mask_folder / f"{name}_mask.jpg"
    if manual_mask_path.exists():
        return np.array(Image.open(manual_mask_path).convert("RGB"))
    return np.zeros((100, 100, 3), dtype=np.uint8)


def get_scaled_dimensions(img: Image.Image, threshold=size_threshold):
    """
    Scale the images so that loading time will be shortened, only do this for images where at least one size is bigger than the threshold (1500 px), and divide both sides by 2
    """
    w, h = img.size
    if w > threshold or h > threshold:
        return (w // 2, h // 2)
    return (w, h)

In [6]:
#This block is for the circle drawing and overlap check

def check_ellipse_overlap(path, annotation):
    """
    Check how much overlap there is between the drawn annotation of the user and the one from the trained model
    """
    # make sure correct images are selected
    name = Path(path).stem
    gt_mask_path = mask_folder_groundtruth / f"{name}_subtracted_mask.jpg"
    # userpred = np.array(Image.open(mask_folder / f"{name}_mask.jpg")) 
    #groundtruth = np.array(Image.open(Path(r"C:\Users\reyna\OneDrive\Documenten\GitHub\Systeemproject\masks_subtracted") / f"{name}_subtracted_mask.jpg"))

    # BUG WHERE IT CRASHES WHEN GT NORMAL FIX
    if gt_mask_path.exists():
        groundtruth_raw = Image.open(gt_mask_path).convert("L")
        #Reshape images
        new_size = get_scaled_dimensions(groundtruth_raw)
        width, height = new_size
        groundtruth_resized = groundtruth_raw.resize(new_size, Image.Resampling.NEAREST)
        groundtruth = np.array(groundtruth_resized)
    else:
        original_img = Image.open(path)
        new_size = get_scaled_dimensions(original_img)
        width, height = new_size
        groundtruth = np.zeros((height, width), dtype=np.uint8)
        

    ellipse_mask = Image.new("L", (width, height), color=0)
    draw = ImageDraw.Draw(ellipse_mask)

    cx, cy = annotation["center"]
    rx = annotation["radius_x"]
    ry = annotation["radius_y"]

    draw.ellipse(
        [(cx - rx, cy - ry), (cx + rx, cy + ry)],
        fill=255,
        outline=255
    )

    userpred = np.array(ellipse_mask)
    
    # Calculate the dice score between the two masks
    teller = 2 * (userpred & groundtruth).sum()
    noemer = userpred.sum() + groundtruth.sum()

    # If someone has (nearly impossible) has the perfect overlap, the dice score is 1 
    if noemer == 0:
        dice_score = 1
    else:
        dice_score = teller/noemer

    #Give feedback based on the dice score
    if dice_score >= 0.95:
        message = f" {dice_score:.3f}. Perfect score!"
    elif dice_score < 0.95 and dice_score >= 0.8:
        message = f"{dice_score:.3f}. Great score!"
    elif dice_score < 0.8 and dice_score >= 0.6:
        message = f"{dice_score:.3f}. Good score."
    elif dice_score < 0.6 and dice_score >= 0.5:
        message = f"{dice_score:.3f}. Adequate score."
    elif dice_score == 0:
        message = f"{dice_score:.3f}. Fail, no overlap detected."
    else:
        message = f"{dice_score:.3f}. Fail, room for improvement."
    return message, dice_score

# Annotation state for 3-click ellipse
annotation_state = {
    "center": None,        # (x, y) set on click 1
    "ry": None,            # vertical radius set on click 2
    "current_image": None  # clean copy of image (Withou annotation)
}

#Dictionairy with the coords of the drawn points
end_coords = {}


def draw_circle(image_input, radio_choice, evt: gr.SelectData):
    """
    Draw the elipse based on 3 clicked points (the middle, the top or bottom (vertical radius) and the most left or right side of the tumor (horizontal radius)
    """
    #If the user classifies the breast as normal, drawing will be disabled because no tumors will be present 
    if radio_choice == "Normal":
        current_path = current_img[-1]
        blank_mask = save_mask(current_path)
        return image_input, "ℹ️ Drawing disabled for Normal images — no tumor to annotate.", gr.update()
    current_path = current_img[-1]    
    x = evt.index[0]
    y = evt.index[1]

    # get current mask (to save at step 1 and 2)
    name = Path(current_path).stem
    manual_mask_path = mask_folder / f"{name}_mask.jpg"
    
    if manual_mask_path.exists():
        current_mask_np = np.array(Image.open(manual_mask_path).convert("RGB"))
    else:
        img_raw = Image.open(current_path)
        new_size = get_scaled_dimensions(img_raw)
        current_mask_np = np.zeros((new_size[1], new_size[0], 3), dtype=np.uint8)
    
    # Click 1: center and return its coordinates
    if annotation_state["center"] is None:
        annotation_state["center"] = (x, y)
        annotation_state["current_image"] = image_input.copy()

        img = Image.fromarray(image_input)
        draw = ImageDraw.Draw(img)
        dot_r = 7
        draw.ellipse(
            [(x - dot_r, y - dot_r), (x + dot_r, y + dot_r)],
            fill="blue", outline="blue"
        )
        coords = f"Step 1/3 — Center set at ({x}, {y}). Now click the TOP or BOTTOM edge of the tumor."
        return np.array(img), coords, gr.update()

    # Click 2: draw vertical radius as a yellow line through the middle point and return its coordinates
    elif annotation_state["ry"] is None:
        cx, cy = annotation_state["center"]
        ry = max(abs(y - cy), 10)
        annotation_state["ry"] = ry

        img = Image.fromarray(annotation_state["current_image"])
        draw = ImageDraw.Draw(img)
        draw.line([(cx, cy - ry), (cx, cy + ry)], fill="yellow", width=3)
        dot_r = 7
        draw.ellipse(
            [(cx - dot_r, cy - dot_r), (cx + dot_r, cy + dot_r)],
            fill="blue", outline="blue"
        )
        coords = f"Step 2/3 — Vertical radius: {ry}px. Now click the LEFT or RIGHT edge of the tumor."
        return np.array(img), coords, gr.update()

    # Click 3: draw horizontal radius as a red line through the middle point and draw the final ellipse based on the 3 clicks
    else:
        cx, cy = annotation_state["center"]
        ry = annotation_state["ry"]
        rx = max(abs(x - cx), 10)

        img = Image.fromarray(annotation_state["current_image"])
        draw = ImageDraw.Draw(img)
        #This is so it is gray if normal
        ellipse_color = "gray" if radio_choice == "Normal" else "green" if radio_choice == "Benign" else "red"

        #Draw ellipse based on the coordinates of the 3 clicks
        draw.ellipse(
            [(cx - rx, cy - ry), (cx + rx, cy + ry)],
            outline=ellipse_color, width=5
        )
        cross = 6
        draw.line([(cx - cross, cy), (cx + cross, cy)], fill=ellipse_color, width=2)
        draw.line([(cx, cy - cross), (cx, cy + cross)], fill=ellipse_color, width=2)

        current_path = current_img[-1]
        if current_path not in end_coords:
            end_coords[current_path] = []
        new_annotation = {"center": (cx, cy), "radius_x": rx, "radius_y": ry}
        end_coords[current_path].append(new_annotation)
        updated_mask = save_mask(current_path)

        if radio_choice == "Normal":
            overlap_result = "ℹ️ No overlap check for Normal images."
        else:
            overlap_result = check_ellipse_overlap(current_path, new_annotation)

        annotation_state["center"] = None
        annotation_state["ry"] = None
        annotation_state["current_image"] = None

        current_annotations = end_coords[current_path]
        coords = f"Ellipses saved!\n"
        for i, ann in enumerate(current_annotations, 1):
            coords += f"Ellipse {i}: Center={ann['center']}, Rx={ann['radius_x']}px, Ry={ann['radius_y']}px\n"

        coords_history[current_path] = coords

        return np.array(img), coords, updated_mask  # ← echte mask bij klik 3


def reset_annotations():
    """
    Clear all drawn ellipses and points of the current image to start over
    """
    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None
    
    current_path = current_img[-1]

    # Delete saved coordinates of the current ellipses    
    if current_path in end_coords:
        del end_coords[current_path]
        
    # Delete saved display history
    if current_path in coords_history:
        coords_history[current_path] = "Annotations cleared"  
        
    #Delete check results
    if current_path in check_results:
        old_result =  check_results[current_path]
        if old_result.startswith("Tumor classification: Correct"):
            check_results[current_path] = "Tumor classification: Correct."
        elif old_result.startswith("Tumor classification: Incorrect"):
            check_results[current_path] = "Tumor classification: Incorrect."

    # Update mask (black with correct scale)
    img_raw = Image.open(current_path)
    new_size = get_scaled_dimensions(img_raw)
    img_resized = img_raw.resize(new_size, Image.Resampling.LANCZOS)

    # Save empty mask
    blank_mask = Image.new("L", new_size, color=0)
    mask_path = mask_folder / (Path(current_path).stem + "_mask.jpg")
    blank_mask.save(mask_path)
    
    updated_mask = np.array(blank_mask.convert("RGB"))

    saved_result = check_results.get(current_path, "")
    return np.array(img_resized), "Annotations cleared", updated_mask, saved_result

def redraw_existing_annotations(img, path, color="green"):
    """
    Redraw elipsis when they were drawn before for the back and next button
    """
    if path in end_coords:
        draw = ImageDraw.Draw(img)
        for annotation in end_coords[path]:
            cx, cy = annotation["center"]
            rx = annotation["radius_x"]
            ry = annotation["radius_y"]
            
            draw.ellipse(
                [(cx - rx, cy - ry), (cx + rx, cy + ry)],
                outline=color,
                width=10
            )
            cross = 6
            draw.line([(cx - cross, cy), (cx + cross, cy)], fill="red", width=2)
            draw.line([(cx, cy - cross), (cx, cy + cross)], fill="red", width=2)
    return np.array(img)

def save_mask(path):
    """
    Save the masks in a folder to be able to extract them later
    """
    original = Image.open(path)
    #rescale the images
    new_size = get_scaled_dimensions(original)
    
    mask = Image.new("L", new_size, color=0)
    draw = ImageDraw.Draw(mask)
    
    if path in end_coords:
        for annotation in end_coords[path]:
            cx, cy = annotation["center"]
            rx = annotation["radius_x"]
            ry = annotation["radius_y"]
            draw.ellipse(
                [(cx - rx, cy - ry), (cx + rx, cy + ry)],
                fill=255,
                outline=255
            )
    
    # Save in mask_folder with the same file name
    mask_path = mask_folder / (Path(path).stem + "_mask.jpg")
    mask.save(mask_path)
    return np.array(mask.convert("RGB"))

def on_radio_change(radio_choice, current_canvas):
    """
    When normal is selected as a classification, change color to gray, empty coords and communicate with user
    """
    path = current_img[-1]
    saved_radio_choices[path] = radio_choice

    annotation_state["center"] = None
    annotation_state["ry"] = None
    annotation_state["current_image"] = None 

    color = "gray" if radio_choice == "Normal" else "green" if radio_choice == "Benign" else "red"
    img = Image.open(path)
    new_size = get_scaled_dimensions(img)
    img = img.resize(new_size, Image.Resampling.LANCZOS)
    redrawn = redraw_existing_annotations(img, path, color=color)

    coords_text = coords_history.get(path, "")
    
    if radio_choice == "Normal":
        original = Image.open(path)
        new_size = get_scaled_dimensions(original)
        original = original.resize(new_size, Image.Resampling.LANCZOS)
        blank = np.zeros((original.height, original.width, 3), dtype=np.uint8)
        return redrawn, blank, "ℹ️ No overlap check for Normal images."
    updated_mask = save_mask(path)
    return redrawn, updated_mask, coords_text
    

In [7]:
# Here all the functions for the hint system

hint_index = {}  # { filename: int }, dit houdt bij welke hint als volgende getoond wordt

def load_excel_full(xlsx_path="Radiology-manual-annotations.xlsx"):
    """
    Load all colomns from the excel for hint generation
    """
    df = pd.read_excel(xlsx_path)
    df["Image_name"] = df["Image_name"].str.strip()
    return df

def load_segmentation_regions(csv_path="Radiology_hand_drawn_segmentations_v2.csv"):
    """
    Returns a dictionairy for {image_stem: region_count} from the CSV.
    """
    df_s = pd.read_csv(csv_path)
    region_map = df_s.groupby("#filename")["region_count"].first()
    return {Path(k).stem: int(v) for k, v in region_map.items()}

def build_hints(filename):
    """
    Builds a list of seperate hints for an image
    """
    row = excel_df[excel_df["Image_name"] == filename]
    if row.empty:
        return []

    row = row.iloc[0]
    hints = []

    # Age hint
    age = row.get("Age", None)
    if pd.notna(age):
        age = int(age)
        if age < 40:
            age_ctx = "younger patient (under 40)"
        elif age < 55:
            age_ctx = "middle-aged patient (40–55)"
        else:
            age_ctx = "older patient (55+)"
        hints.append(f"Patient age: {age} ({age_ctx})")
    

    # Breast density hint
    density = str(row.get("Breast density (ACR)", "_")).strip()
    density_map = {
        "A": "A – Almost entirely fatty (easy to read)",
        "B": "B – Scattered fibroglandular density",
        "C": "C – Heterogeneously dense (may obscure small masses)",
        "D": "D – Extremely dense (reduces sensitivity)",
    }
    if density in density_map:
        hints.append(f"Breast density: {density_map[density]}")

    # View hint  # mogelijk nuttig, besproken met 1 geneeskunde student als bron
    view = str(row.get("View", "")).strip()
    view_desc = {
        "CC": "CC (Craniocaudal) — top-down view",
        "MLO": "MLO (Mediolateral Oblique) — angled side view",
    }
    if view in view_desc:
        hints.append(f"View: {view_desc[view]}")

    # Regions / tumors hint (from CSV)
    region_count = region_counts.get(filename, None)
    if region_count is not None:
        if region_count == 1:
            hints.append("Regions in ground truth: 1, single finding expected")
        elif region_count <= 4:
            hints.append(f"Regions in ground truth: {region_count}, multiple findings, annotate each one!")
        else:
            hints.append(f"Regions in ground truth: {region_count}, complex case with many findings!")

    # Image type hint  # NIET NUTTIG: ALLEEN SUBTRACTED IMAGES DUS DEZE UITGESCHAKELD
    #img_type = str(row.get("Type", "")).strip()
    #if img_type == "DM":
    #    hints.append("Image type: Digital Mammography (DM), standard X-ray density view")
    #elif img_type == "CESM":
    #    hints.append("Image type: Contrast-Enhanced Spectral Mammography (CESM), enhancement indicates vascularity")

    # BIRADS hint
    birads_raw = str(row.get("BIRADS", "")).strip()
    birads_vals = [b.strip() for b in birads_raw.split("$") if b.strip()]
    birads_desc = {
        "1": "1 – Negative (routine screening)",
        "2": "2 – Benign finding",
        "3": "3 – Probably benign (short follow-up suggested)",
        "4": "4 – Suspicious (biopsy recommended)",
        "5": "5 – Highly suggestive of malignancy",
        "6": "6 – Known biopsy-proven malignancy",
    }
    if birads_vals:
        birads_strs = [birads_desc.get(b, f"BIRADS {b}") for b in birads_vals]
        hints.append(f"BIRADS: {' + '.join(birads_strs)}")

    # Side hint # wss niet nuttig, besproken met dezelfde 1e geneeskundestudent als bron
    #side = str(row.get("Side", "")).strip()
    #if side in ("L", "R"):
    #    hints.append(f"Side: {'Left' if side == 'L' else 'Right'} breast")

    return hints

def get_hint():
    """
    Gives hints one by one but does show all previous given hints
    """
    if not current_img:
        return "No image loaded!"

    filename = Path(current_img[-1]).stem
    hints = build_hints(filename)

    if not hints:
        return f"No hints available for '{filename}'."

    idx = hint_index.get(filename, 0)
    if idx >= len(hints):
        return "\n\n".join(hints) + f"\n\nAll hints shown ({len(hints)}/{len(hints)})"

    hint_index[filename] = idx + 1
    shown = hints[:idx + 1]  #all hints up until now 
    return "\n\n".join(shown) + f"\n\n({idx + 1}/{len(hints)})"

def reset_hint():
    """
    Erases the hint index for the current image (for next / back)
    """
    if current_img:
        filename = Path(current_img[-1]).stem
        hint_index.pop(filename, None)

def get_shown_hints():
    """
    Returns all shown hints for the current image
    """
    if not current_img:
        return ""
    filename = Path(current_img[-1]).stem
    hints = build_hints(filename)
    idx = hint_index.get(filename, 0)
    if idx == 0:
        return ""
    shown = hints[:idx]
    return "\n\n".join(shown) + f"\n\n({idx}/{len(hints)})"


In [8]:
#This block is for the demo
#This is to select the first image to work with
first_image = load_image()
#This is too call up the preped excel sheet
excel_annotations = load_excel_annotations()
excel_df = load_excel_full()          
region_counts = load_segmentation_regions()  

#Create the demo
with gr.Blocks() as demo:
    #Give the buttons different colors
    gr.HTML("""
    <style>
    .green-btn > button, button.green-btn {
        background: #2ecc71 !important;
        color: white !important;
        border: none !important;
    }
    .green-btn > button:hover, button.green-btn:hover {
        background: #27ae60 !important;
    }
    .blue-btn > button, button.blue-btn {
        background: #3498db !important;
        color: white !important;
        border: none !important;
        }
    .blue-btn > button:hover, button.blue-btn:hover {
        background: #2980b9 !important;
        }
    </style>
    """)

    #Define the gif for the tutorial pop up
    gif = "tut.gif"

    with Modal(visible=True) as instructions_modal:
        # The tutorial pop up with the implemented give
        gr.Markdown("# Instructions")
        gr.Markdown(
            """
            This application is designed to analyze mammograms and annotate tumors. Below you'll find the instructions to proceed correctly:
            ### How to annotate (see the gif):
            * **Click 1** – Click on the **exact center** of the tumor.
            * **Click 2** – Click on the **top or bottom edge** of the tumor to determine the vertical size.
            * **Click 3** – Click on the **left or right edge** of the tumor to complete the ellipse.

            ### Other Features:
            * **Hint:** Get additional patient information step-by-step (such as age or breast density) if you get stuck.
            * **Show answer:** Compare your hand-drawn ellipse directly with the correct answer.
            """
        )
        gr.Image(value=gif, label="How to select the tumor(s)", interactive=False)
        close_popup_btn = gr.Button("Start the application", variant="primary", elem_classes=["blue-btn"])
        
        # Closing logic for the pop up
        close_popup_btn.click(lambda: Modal(visible=False), None, instructions_modal)
    
    img_state = gr.State(value=first_image)
    
    #Add the instructions
    gr.Markdown(
        "**How to annotate:** "
        "**Click 1** – center of the tumor &nbsp;|&nbsp; "
        "**Click 2** – top or bottom edge &nbsp;|&nbsp; "
        "**Click 3** – left or right edge"

    )

    tutorial_btn = gr.Button("Show tutorial")
    progress_output = gr.Textbox(value=get_progress(), label="Progress", interactive=False)


    #Save original image in a state
    with gr.Row():
        #Create a variable with the image, value is the np array, type is thus numpy, webcam and other features are disabled and it is intractive.
        scan_canvas = gr.Image(value=first_image, type="numpy", interactive=False, sources=[] ,width=300, height=300)

        with gr.Column():
            #coordes output + image output
            coords_output = gr.Textbox(label="Selected coordinates")
            #Adding the buttons for what type!
            radio = gr.Radio(["Benign", "Malignant", "Normal"], value="Benign", label = "Choose the type")
            #Adding the check answer button and output
            with gr.Row():
                check_btn = gr.Button("Check")
                show_btn = gr.Button("Show Answer")
                hint_btn = gr.Button("Hint")
            check_output = gr.Textbox(label="Check result", lines=2, max_lines=5, autoscroll=False)
            show_output = gr.Textbox(label="Correct Answer", lines = 1, max_lines = 1, autoscroll=False)
            hint_output = gr.Textbox(label="Hint", lines=7, interactive=False)
            
    #Create the next button
    with gr.Row():
        back_btn = gr.Button("<- Back", visible=False)
        reset_btn = gr.Button("Reset annotations")
        next_btn = gr.Button("Next ->", visible=len(images_path) > 0)

    # Mask comparison section
    with gr.Row():
        show_masks_btn = gr.Button("Show tumor area", elem_classes=["green-btn"])
        restart_btn = gr.Button("🔄 New session", elem_classes=["blue-btn"])
        hide_masks_btn = gr.Button("Hide masks", variant="stop", visible=False)

    with gr.Row(visible=False) as mask_row:
        manual_mask_img = gr.Image(label="Your selected tumor area", type="numpy", sources=[], width=300, height=300)
        gt_mask_img = gr.Image(label="Answer", type="numpy", sources=[], width=300, height=300)

    def hide_masks():
        """
        Hides the shown masks (drawn and ground truth)
        """
        return gr.update(visible=False), gr.update(visible=False)

    def next_image_and_check_last():
        #reset_hint()
        img, back, next_update, saved, radio_update = next_image()
        saved_coords = coords_history.get(current_img[-1], "") if current_img else ""
        # Get the manual mask of the current image 
        if current_img:
            name = Path(current_img[-1]).stem
            manual_mask_path = mask_folder / f"{name}_mask.jpg"
            if manual_mask_path.exists():
                updated_manual_mask = np.array(Image.open(manual_mask_path).convert("RGB"))
            else:
                updated_manual_mask = np.zeros((100, 100, 3), dtype=np.uint8)
        else:
            updated_manual_mask = np.zeros((100, 100, 3), dtype=np.uint8)
            
        return img, back, gr.update(visible=bool(images_path)), saved, saved_coords, "", updated_manual_mask, radio_update

   
    
    #print the image            
    scan_canvas.select(
    fn=draw_circle, 
    inputs=[scan_canvas, radio], 
    outputs=[scan_canvas, coords_output, manual_mask_img])
    #Get all buttons to work using all predefined functions
    next_btn.click(
        fn=next_image_and_check_last,
        inputs=[],
        outputs=[scan_canvas, back_btn, next_btn, check_output, coords_output, hint_output, manual_mask_img, radio]
    ).then(fn=hide_masks, inputs=[], outputs=[mask_row, hide_masks_btn]
    ).then(fn=lambda: "", outputs=[show_output]
    ).then(fn=get_shown_hints, outputs=[hint_output]
    ).then(fn=get_progress, inputs=[], outputs=[progress_output]
    ).then(fn=get_current_gt_mask, outputs=[gt_mask_img])

    back_btn.click(
        fn=back_image, 
        inputs=[], 
        outputs=[scan_canvas, back_btn, next_btn, check_output, coords_output, manual_mask_img, radio]
    ).then(fn=hide_masks, inputs=[], outputs=[mask_row, hide_masks_btn]
    ).then(fn=lambda: ("",""), outputs=[show_output, hint_output]
    ).then(fn=get_shown_hints, outputs=[hint_output] 
    ).then(fn=get_progress, inputs=[], outputs=[progress_output]
    ).then(fn=get_current_gt_mask, outputs=[gt_mask_img])

    reset_btn.click(
        fn=reset_annotations, inputs=[], outputs=[scan_canvas, coords_output, manual_mask_img, check_output]
    )

    check_btn.click(
        fn=check_annotation, inputs=[radio], outputs=[check_output]
    ).then(fn=get_progress, inputs=[], outputs=[progress_output])

    show_btn.click(
        fn=show_annotation, inputs=[], outputs=[show_output]
    )

    restart_btn.click(
        fn=restart_session, inputs=[],
        outputs=[scan_canvas, back_btn, next_btn, restart_btn, coords_output, check_output, show_output, progress_output, hint_output, radio]
        ).then(fn=hide_masks, inputs=[], outputs=[mask_row, hide_masks_btn]
        ).then(fn=get_current_gt_mask, outputs=[gt_mask_img]
        ).then(fn=get_current_manual_mask, outputs=[manual_mask_img]
    )

    show_masks_btn.click(
        fn=show_masks, inputs=[],
        outputs=[manual_mask_img, gt_mask_img, mask_row, hide_masks_btn]
    )

    hide_masks_btn.click(
        fn=hide_masks, inputs=[],
        outputs=[mask_row, hide_masks_btn]
    )

    radio.change(
        fn=on_radio_change,
        inputs=[radio, scan_canvas],
        outputs=[scan_canvas, manual_mask_img, coords_output]
    )
    hint_btn.click(
        fn=get_hint,
        inputs=[],
        outputs=[hint_output]
    )
    tutorial_btn.click(
        lambda: Modal(visible=True), None, instructions_modal
    )

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
